# Lean-RGC v53 Bivariate + Face Taxonomy Colab

This notebook clones the public repo, installs Lean/Python dependencies, runs a bivariate kernel-RPC experiment, and builds the finite dual-exposed face taxonomy layer.

The important pass gates are `stateful_kernel_rpc_audit=true`, `source_check_calls=0`, `baseline_missing=0`, nondegenerate bivariate rows/columns, and nonempty `dual_face_taxonomy` / `face_concept_lattice` artifacts.

In [ ]:
%%bash
set -euo pipefail
REPO_URL="${REPO_URL:-https://github.com/abhorrence-of-Gods/lean-rgc-automation-stack-v51.git}"
REPO_DIR="${REPO_DIR:-/content/lean-rgc-automation-stack-v51}"
if [ ! -d "$REPO_DIR/.git" ]; then
  git clone "$REPO_URL" "$REPO_DIR"
else
  cd "$REPO_DIR"
  git pull --ff-only
fi
cat > /content/lean_rgc_env.sh <<EOF
export REPO_URL="$REPO_URL"
export REPO_DIR="$REPO_DIR"
EOF
source /content/lean_rgc_env.sh
cd "$REPO_DIR"
git rev-parse --short HEAD
git status --short

In [ ]:
%%bash
set -euo pipefail
source /content/lean_rgc_env.sh
cd "$REPO_DIR"
python -m pip install -U pip
python -m pip install -e . pytest
python -m lean_rgc.cli face-taxonomy --help | head -n 40

In [ ]:
%%bash
set -euo pipefail
if [ ! -f "$HOME/.elan/env" ]; then
  curl https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh -sSf | sh -s -- -y --default-toolchain stable
fi
source "$HOME/.elan/env"
lean --version
lake --version

In [ ]:
%%bash
set -euo pipefail
source /content/lean_rgc_env.sh
source "$HOME/.elan/env"
cd "$REPO_DIR/lean_project_template"
lake update || true
lake build

In [ ]:
%%bash
set -euo pipefail
source /content/lean_rgc_env.sh
cd "$REPO_DIR"
python -m pytest -q tests/test_v51_bivariate_contextual_quotient.py tests/test_v52_kernel_context_cache.py tests/test_v52_face_taxonomy.py

## Expanded row universe

This creates a small but nontrivial row universe. It intentionally includes structural, normalization, and arithmetic-like rows so the face taxonomy has something to separate. This is still a chart-level pilot, not canonical typed premise-use extraction.

In [ ]:
%%bash
set -euo pipefail
source /content/lean_rgc_env.sh
cd "$REPO_DIR"
cat > examples/core_tactics_rows_v53.jsonl <<'EOF'
{"action_id":"trivial","tactic":"trivial"}
{"action_id":"rfl","tactic":"rfl"}
{"action_id":"exact_rfl","tactic":"exact rfl"}
{"action_id":"simp","tactic":"simp"}
{"action_id":"simp_all","tactic":"simp_all"}
{"action_id":"assumption","tactic":"assumption"}
{"action_id":"constructor","tactic":"constructor"}
{"action_id":"constructor_assumption","tactic":"constructor <;> assumption"}
{"action_id":"intro","tactic":"intro"}
{"action_id":"intros","tactic":"intros"}
{"action_id":"intro_simp","tactic":"intro n; simp"}
{"action_id":"intro_assumption","tactic":"intro h; assumption"}
{"action_id":"intro_rfl","tactic":"intro x; rfl"}
{"action_id":"intro_exact_rfl","tactic":"intro x; exact rfl"}
{"action_id":"intro_simp_all","tactic":"intro x; simp_all"}
{"action_id":"constructor_simp","tactic":"constructor <;> simp"}
{"action_id":"constructor_rfl","tactic":"constructor <;> rfl"}
{"action_id":"norm_num","tactic":"norm_num"}
{"action_id":"decide","tactic":"decide"}
{"action_id":"omega","tactic":"omega"}
{"action_id":"simp_arith","tactic":"simp; omega"}
{"action_id":"intros_simp_all","tactic":"intros; simp_all"}
{"action_id":"apply_assumption","tactic":"apply ‹_›"}
{"action_id":"exact_assumption","tactic":"exact ‹_›"}
EOF
wc -l examples/core_tactics_rows_v53.jsonl
head -n 8 examples/core_tactics_rows_v53.jsonl

## Run bivariate kernel-RPC audit + face taxonomy

This is the main smoke/medium command. It uses the kernel context cache path, so contextual audits should report `source_check_calls=0` and positive cache hits.

In [ ]:
%%bash
set -euo pipefail
source /content/lean_rgc_env.sh
source "$HOME/.elan/env"
cd "$REPO_DIR"
RUN="runs/colab_v53_bivariate_face_taxonomy_$(date +%Y%m%d_%H%M%S)"
mkdir -p "$(dirname "$RUN")"
lean-rgc pipeline \
  --tasks examples/core_theorems.jsonl \
  --actions examples/core_tactics_rows_v53.jsonl \
  --out "$RUN" \
  --audit-mode server \
  --server-backend native \
  --native-exec-mode kernel_rpc \
  --server-no-fallback \
  --lean-cmd "lake env lean" \
  --workdir lean_project_template \
  --import-mode core \
  --max-actions 400 \
  --premise-contextual-bivariate \
  --premise-contextual-quotient \
  --premise-contextual-max-premises 24 \
  --premise-contextual-max-left 6 \
  --premise-contextual-max-right 10 \
  --bivariate-audit-budget 1024 \
  --premise-contextual-baseline-required \
  --skip-vacuous-premise-quotient \
  --repair-face-ledger \
  --face-taxonomy \
  2>&1 | tee "$RUN.log"
echo "$RUN" > /tmp/lean_rgc_last_run.txt
echo "RUN=$RUN"

## Check bivariate and taxonomy artifacts

The first goal is not high theorem solve rate. The goal is a nondegenerate bivariate transcript and nonempty finite face taxonomy.

In [ ]:
from pathlib import Path
import json
from collections import Counter

run = Path(Path('/tmp/lean_rgc_last_run.txt').read_text().strip())
print('RUN =', run)

def load_json(path):
    p = Path(path)
    if not p.exists():
        print('MISSING', p)
        return {}
    return json.loads(p.read_text())

def load_jsonl(path):
    p = Path(path)
    if not p.exists():
        print('MISSING', p)
        return []
    return [json.loads(x) for x in p.read_text().splitlines() if x.strip()]

reports = {
    'main_summary': run / 'audit/server_summary.json',
    'ctx_summary': run / 'premise_contextual_audit/server_summary.json',
    'fp_report': run / 'premise_contextual/premise_contextual_fingerprint_report.json',
    'quot_report': run / 'premise_contextual/premise_quotient_report.json',
    'val_report': run / 'premise_contextual/premise_quotient_validation_report.json',
    'face_report': run / 'premise_contextual/repair_face_ledger_report.json',
    'tax_report': run / 'premise_contextual/face_taxonomy/dual_face_taxonomy_report.json',
}

keys = [
    'n_responses','n_failures','statuses','success_rate','timeout_rate','stateful_kernel_rpc_audit',
    'kernel_context_cache','context_cache_hits','source_check_calls',
    'n_fingerprints','n_unique_premise_use_ids','n_unique_context_pairs',
    'baseline_missing','row_degenerate','column_degenerate',
    'n_premise_use_rows','n_classes','n_faces',
    'n_rows','n_properties','n_concepts','n_taxonomy_faces','n_retrieval_allowed_faces',
    'dual_source_counts','human_name_counts','retrieval_blocker_counts',
]

for name, path in reports.items():
    obj = load_json(path)
    print('\n==', name, path)
    print(json.dumps({k: obj.get(k) for k in keys if k in obj}, indent=2, ensure_ascii=False))

print('\n== Line counts ==')
for rel in [
    'premise_contextual/premise_use_rows.jsonl',
    'premise_contextual/separator_contexts.jsonl',
    'premise_contextual/premise_contextual_candidates.jsonl',
    'premise_contextual/bivariate_contextual_scheduled_actions.jsonl',
    'premise_contextual/premise_contextual_fingerprints.jsonl',
    'premise_contextual/premise_quotient_classes.jsonl',
    'premise_contextual/repair_faces.jsonl',
    'premise_contextual_audit/contextual_plan_transitions.jsonl',
    'premise_contextual_audit/kernel_context_state_cache.jsonl',
    'premise_contextual/face_taxonomy/row_property_incidence.jsonl',
    'premise_contextual/face_taxonomy/face_concept_lattice.jsonl',
    'premise_contextual/face_taxonomy/dual_face_taxonomy.jsonl',
    'premise_contextual/face_taxonomy/row_face_memberships.jsonl',
    'premise_contextual/face_taxonomy/taxonomy_name_suggestions.jsonl',
    'premise_contextual/face_taxonomy/retrieval_allowed_faces.jsonl',
]:
    print(rel, len(load_jsonl(run / rel)))

validation = load_jsonl(run / 'premise_contextual/premise_quotient_validation_rows.jsonl')
print('\n== Validation statuses ==')
print(Counter(v.get('validation_status') for v in validation))

taxonomy = load_jsonl(run / 'premise_contextual/face_taxonomy/dual_face_taxonomy.jsonl')
print('\n== Taxonomy sample ==')
for row in taxonomy[:10]:
    print(json.dumps({
        'taxonomy_face_id': row.get('taxonomy_face_id'),
        'dual_source': row.get('dual_source'),
        'human_name': (row.get('human_name') or {}).get('suggested'),
        'rows': (row.get('minimal_support') or {}).get('rows'),
        'carrier_status': (row.get('status') or {}).get('carrier_status'),
        'retrieval_allowed': (row.get('status') or {}).get('retrieval_allowed'),
        'blockers': (row.get('status') or {}).get('retrieval_blockers'),
    }, ensure_ascii=False))

fp = load_json(reports['fp_report'])
ctx = load_json(reports['ctx_summary'])
tax = load_json(reports['tax_report'])
checks = {
    'kernel_context_cache_true': ctx.get('kernel_context_cache') is True,
    'source_check_calls_zero': ctx.get('source_check_calls') == 0,
    'context_cache_hits_positive': (ctx.get('context_cache_hits') or 0) > 0,
    'baseline_missing_zero': fp.get('baseline_missing') == 0,
    'row_not_degenerate': fp.get('row_degenerate') is False,
    'column_not_degenerate': fp.get('column_degenerate') is False,
    'fingerprints_gt_5': (fp.get('n_fingerprints') or 0) > 5,
    'taxonomy_faces_positive': (tax.get('n_taxonomy_faces') or 0) > 0,
    'concepts_positive': (tax.get('n_concepts') or 0) > 0,
}
print('\n== Pass/Fail Gate ==')
print(json.dumps(checks, indent=2, ensure_ascii=False))
print('\nRESULT:', 'PASS' if all(checks.values()) else 'CHECK FAILED')

## Optional: build taxonomy on an existing run

Use this if you already have a previous bivariate run and only want to rebuild the taxonomy layer.

In [ ]:
%%bash
set -euo pipefail
source /content/lean_rgc_env.sh
cd "$REPO_DIR"
RUN="$(cat /tmp/lean_rgc_last_run.txt)"
lean-rgc face-taxonomy \
  --fingerprints "$RUN/premise_contextual/premise_contextual_fingerprints.jsonl" \
  --classes "$RUN/premise_contextual/premise_quotient_classes.jsonl" \
  --validation "$RUN/premise_contextual/premise_quotient_validation_rows.jsonl" \
  --repair-faces "$RUN/premise_contextual/repair_faces.jsonl" \
  --out "$RUN/premise_contextual/face_taxonomy_rebuilt"
find "$RUN/premise_contextual/face_taxonomy_rebuilt" -maxdepth 1 -type f -print

In [ ]:
%%bash
set -euo pipefail
source /content/lean_rgc_env.sh
cd "$REPO_DIR"
RUN="$(cat /tmp/lean_rgc_last_run.txt)"
ZIP="/content/$(basename "$RUN")_artifacts.zip"
zip -qr "$ZIP" "$RUN" "$RUN.log"
echo "$ZIP"